# TinyAudioTransformer: Phase 2b (combine the winners, test on held-out noise)

**Notebook settings:** Accelerator = **GPU T4 x2**, Internet = **On**.

**Run it with Save Version → Save & Run All (Commit).** About an hour. Results land in the Output tab as `phase2b_results.zip`.

Phase 2 found two clear wins (time shift +1.0, SpecAugment +1.1, every seed). Mixup and label smoothing made the model underconfident, background noise cost 1.3 points *on the clean test*, and all augmentations together underfit. Phase 2b asks:
1. **Do the winners add up?** `shift + specaug`.
2. **Do the others help once the winners are in?** `+ mixup`, `+ ls`, `+ noise`, each compared with `shift + specaug`.
3. **What does noise training buy?** Every checkpoint is re-scored with **held-out** background noise (last 20% of each recording, never trained on) at 20 / 10 / 5 / 0 dB.

`none` and `noise` are re-run here, so this notebook is self-contained and doubles as a reproducibility check against Phase 2. 6 configs × 3 seeds = 18 runs.

Generated by `kaggle/build_notebook.py`. Edit the repo files, not these cells.

In [ ]:
import os, sys, json, pathlib
os.makedirs("/kaggle/working/tat/tests", exist_ok=True)
os.chdir("/kaggle/working/tat")
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| GPUs:", torch.cuda.device_count())
try:
    import transformers; print("transformers", transformers.__version__)
except ImportError:
    print("transformers missing → AST parity tests will be skipped")
!pip install -q pytest

## Source files

In [ ]:
%%writefile /kaggle/working/tat/features.py
# features.py
# Kaldi-compatible log-mel filterbank in plain torch, matching
# torchaudio.compliance.kaldi.fbank(wav, sample_frequency=16000, window_type="hanning", num_mel_bins=128)
# which is what AST (and HF's ASTFeatureExtractor) use. Owning it removes the torchaudio dependency,
# works on batches/GPU, and gives one reference to port when the feature pipeline moves to the browser.
# tests/test_features.py checks it against torchaudio and the HF extractor.
import math

import torch
import torch.nn.functional as F

SAMPLE_RATE = 16000
FRAME_LENGTH = 400      # 25 ms
FRAME_SHIFT = 160       # 10 ms
N_FFT = 512             # frame length rounded up to a power of two
PREEMPHASIS = 0.97
LOW_FREQ = 20.0
EPS = torch.finfo(torch.float32).eps


def _mel(f):
    return 1127.0 * torch.log1p(f / 700.0)


def mel_banks(n_mels=128, sample_rate=SAMPLE_RATE, n_fft=N_FFT, low_freq=LOW_FREQ):
    # Triangles evenly spaced on the mel scale, evaluated at FFT bin centres; Nyquist bin gets weight 0
    mel_lo = 1127.0 * math.log1p(low_freq / 700.0)
    mel_hi = 1127.0 * math.log1p((sample_rate / 2) / 700.0)
    delta = (mel_hi - mel_lo) / (n_mels + 1)

    b = torch.arange(n_mels).unsqueeze(1)
    left, center, right = (mel_lo + (b + k) * delta for k in (0.0, 1.0, 2.0))
    mel = _mel(sample_rate / n_fft * torch.arange(n_fft // 2)).unsqueeze(0)

    up = (mel - left) / (center - left)
    down = (right - mel) / (right - center)
    banks = torch.clamp(torch.min(up, down), min=0.0)        # [n_mels, n_fft/2]
    return F.pad(banks, (0, 1))                               # [n_mels, n_fft/2 + 1]


def fbank(wav, n_mels=128, n_frames=None):
    """wav: float waveform in [-1, 1], shape [N] or [B, N] at 16 kHz.
    Returns log-mel [T, n_mels] (or [B, T, n_mels]); if n_frames is given, zero-pads/crops time to it."""
    squeeze = wav.dim() == 1
    wav = wav.reshape(-1, wav.shape[-1]).float()

    frames = wav.unfold(-1, FRAME_LENGTH, FRAME_SHIFT)                         # [B, T, 400], snip_edges=True
    frames = frames - frames.mean(dim=-1, keepdim=True)                         # remove DC per frame
    prev = torch.cat((frames[..., :1], frames[..., :-1]), dim=-1)
    frames = frames - PREEMPHASIS * prev                                        # pre-emphasis
    frames = frames * torch.hann_window(FRAME_LENGTH, periodic=False, device=wav.device)
    frames = F.pad(frames, (0, N_FFT - FRAME_LENGTH))

    power = torch.fft.rfft(frames).abs().pow(2)                                 # [B, T, 257]
    mel = power @ mel_banks(n_mels).to(wav.device).T                            # [B, T, n_mels]
    out = torch.clamp(mel, min=EPS).log()

    if n_frames is not None:
        out = pad_or_crop(out, n_frames)
    return out[0] if squeeze else out


def num_frames(n_samples):
    # Frames kaldi produces with snip_edges=True: 16000 samples → 98
    return 1 + (n_samples - FRAME_LENGTH) // FRAME_SHIFT


def pad_or_crop(x, n_frames):
    # x: [..., T, F]; zero-pad at the end of time like AST's ZeroPad2d, or crop
    T = x.shape[-2]
    if T < n_frames:
        return F.pad(x, (0, 0, 0, n_frames - T))
    return x[..., :n_frames, :]

In [ ]:
%%writefile /kaggle/working/tat/patch_embed.py
# patch_embed.py
import torch
import torch.nn as nn


def num_patches_1d(size, patch, stride):
    # Number of conv windows along one axis (no padding), same formula as AST's get_shape
    return (size - patch) // stride + 1


class PatchEmbed(nn.Module):
    """Spectrogram -> sequence of patch embeddings.

    Input follows the AST convention: [B, T, F] (time frames, mel bins), which is also
    what kaldi fbank returns. With stride < patch the patches overlap (AST: 16x16, stride 10).
    """

    def __init__(self, n_mels=128, n_frames=128, patch_size=16, patch_stride=10, embed_dim=128):
        super().__init__()
        self.n_mels = n_mels
        self.n_frames = n_frames
        self.grid = (num_patches_1d(n_mels, patch_size, patch_stride),
                     num_patches_1d(n_frames, patch_size, patch_stride))
        self.num_patches = self.grid[0] * self.grid[1]

        self.proj = nn.Conv2d(1, embed_dim, kernel_size=patch_size, stride=patch_stride)

    def forward(self, x):
        # x: [B, T, F]
        if x.shape[1:] != (self.n_frames, self.n_mels):
            raise ValueError(f"expected input [B, {self.n_frames}, {self.n_mels}], got {list(x.shape)}")
        x = x.unsqueeze(1).transpose(2, 3)  # → [B, 1, F, T]
        x = self.proj(x)                    # → [B, embed_dim, F', T']
        x = x.flatten(2).transpose(1, 2)    # → [B, F'*T', embed_dim]  (frequency-major order, as in AST)
        return x

In [ ]:
%%writefile /kaggle/working/tat/transformer_block.py
# transformer_block.py
import torch
import torch.nn as nn


class TransformerEncoderBlock(nn.Module):
    """Pre-norm ViT block: x + Attn(LN(x)), then x + MLP(LN(x))."""

    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.1, attn_dropout=0.0, ln_eps=1e-6):
        super().__init__()
        hidden = int(embed_dim * mlp_ratio)

        self.norm1 = nn.LayerNorm(embed_dim, eps=ln_eps)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=attn_dropout, batch_first=True)
        self.drop1 = nn.Dropout(dropout)

        self.norm2 = nn.LayerNorm(embed_dim, eps=ln_eps)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        h = self.norm1(x)  # normalise once, use as q, k and v
        x = x + self.drop1(self.attn(h, h, h, need_weights=False)[0])
        x = x + self.mlp(self.norm2(x))
        return x

In [ ]:
%%writefile /kaggle/working/tat/model.py
# model.py
from dataclasses import dataclass, asdict

import torch
import torch.nn as nn

from patch_embed import PatchEmbed
from transformer_block import TransformerEncoderBlock


@dataclass
class TATConfig:
    n_mels: int = 128
    n_frames: int = 128          # 1 s clip = 98 kaldi frames, zero-padded to 128 (AST speech-commands recipe)
    patch_size: int = 16
    patch_stride: int = 10       # < patch_size → overlapping patches (AST); set to 16 for ViT-style
    embed_dim: int = 128
    depth: int = 4
    num_heads: int = 4
    mlp_ratio: float = 4.0
    num_classes: int = 35
    dropout: float = 0.1
    attn_dropout: float = 0.0
    num_prefix_tokens: int = 1   # 1 = [CLS]; 2 = [CLS]+[DIST] (AST/DeiT), pooled as their mean
    head_norm: bool = False      # extra LayerNorm inside the head (AST's ASTMLPHead has one)
    ln_eps: float = 1e-6


class FeatureNorm(nn.Module):
    """(x - mean) / (2 * std), as in AST. Stats live in the model so they ship with the checkpoint."""

    def __init__(self, mean=0.0, std=0.5):
        super().__init__()
        self.register_buffer("mean", torch.tensor(float(mean)))
        self.register_buffer("std", torch.tensor(float(std)))

    def set_stats(self, mean, std):
        self.mean.fill_(float(mean))
        self.std.fill_(float(std))

    def forward(self, x):
        return (x - self.mean) / (2 * self.std)


class TinyAudioTransformer(nn.Module):
    def __init__(self, cfg: TATConfig = TATConfig()):
        super().__init__()
        self.cfg = cfg
        self.norm_in = FeatureNorm()

        self.patch_embed = PatchEmbed(cfg.n_mels, cfg.n_frames, cfg.patch_size, cfg.patch_stride, cfg.embed_dim)
        n_tokens = self.patch_embed.num_patches + cfg.num_prefix_tokens

        self.prefix_tokens = nn.Parameter(torch.zeros(1, cfg.num_prefix_tokens, cfg.embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_tokens, cfg.embed_dim))
        self.pos_drop = nn.Dropout(cfg.dropout)

        self.blocks = nn.Sequential(*[
            TransformerEncoderBlock(cfg.embed_dim, cfg.num_heads, cfg.mlp_ratio,
                                    cfg.dropout, cfg.attn_dropout, cfg.ln_eps)
            for _ in range(cfg.depth)
        ])
        self.norm = nn.LayerNorm(cfg.embed_dim, eps=cfg.ln_eps)
        self.head_norm = nn.LayerNorm(cfg.embed_dim, eps=cfg.ln_eps) if cfg.head_norm else nn.Identity()
        self.head = nn.Linear(cfg.embed_dim, cfg.num_classes)

        self.apply(self._init_weights)
        nn.init.trunc_normal_(self.prefix_tokens, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    @staticmethod
    def _init_weights(m):
        # ViT/DeiT init: small truncated normal for linears, identity LayerNorms
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
        elif isinstance(m, nn.MultiheadAttention):
            nn.init.trunc_normal_(m.in_proj_weight, std=0.02)
            nn.init.zeros_(m.in_proj_bias)

    def forward(self, x):
        # x: raw log-mel fbank [B, T, F]
        B = x.shape[0]
        x = self.norm_in(x)
        x = self.patch_embed(x)                                 # → [B, N, D]

        prefix = self.prefix_tokens.expand(B, -1, -1)           # → [B, P, D]
        x = torch.cat((prefix, x), dim=1) + self.pos_embed      # → [B, P+N, D]
        x = self.pos_drop(x)

        x = self.blocks(x)
        x = self.norm(x)

        pooled = x[:, :self.cfg.num_prefix_tokens].mean(dim=1)  # [CLS] (or mean of [CLS],[DIST])
        return self.head(self.head_norm(pooled))

    def config_dict(self):
        return asdict(self.cfg)

In [ ]:
%%writefile /kaggle/working/tat/baselines.py
# baselines.py
# A small CNN on the same features. If the transformer cannot beat this, the problem is the transformer
# (or its training), not the data pipeline.
from dataclasses import dataclass, asdict

import torch.nn as nn

from model import FeatureNorm


@dataclass
class CNNConfig:
    channels: tuple = (32, 64, 128, 128)
    num_classes: int = 35
    dropout: float = 0.1


class CNNBaseline(nn.Module):
    def __init__(self, cfg: CNNConfig = CNNConfig()):
        super().__init__()
        self.cfg = cfg
        self.norm_in = FeatureNorm()
        layers, c_in = [], 1
        for c in cfg.channels:
            layers += [nn.Conv2d(c_in, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(), nn.MaxPool2d(2)]
            c_in = c
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(cfg.dropout),
                                  nn.Linear(c_in, cfg.num_classes))

    def forward(self, x):
        # x: raw log-mel fbank [B, T, F]
        x = self.norm_in(x).unsqueeze(1)  # → [B, 1, T, F]
        return self.head(self.features(x))

    def config_dict(self):
        return asdict(self.cfg)

In [ ]:
%%writefile /kaggle/working/tat/dataset.py
# dataset.py
# Speech Commands v2, 35 words, official split (validation_list.txt / testing_list.txt).
# The official lists are speaker-disjoint: the part of the filename before "_nohash_" is the speaker id.
import tarfile
import urllib.request
import wave
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset

from features import SAMPLE_RATE

SC_V2_URL = "http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz"
EXPECTED_SIZES = {"train": 84843, "val": 9981, "test": 11005}
CLIP_SAMPLES = SAMPLE_RATE  # every clip is zero-padded / cropped to 1 s


def download_speech_commands(root):
    root = Path(root)
    if (root / "testing_list.txt").exists():
        return root
    root.mkdir(parents=True, exist_ok=True)
    archive = root / "speech_commands_v0.02.tar.gz"
    if not archive.exists():
        print(f"Downloading {SC_V2_URL} (~2.3 GB)")
        urllib.request.urlretrieve(SC_V2_URL, archive)
    print("Extracting")
    with tarfile.open(archive) as tar:
        kwargs = {"filter": "data"} if hasattr(tarfile, "data_filter") else {}
        tar.extractall(root, **kwargs)
    archive.unlink()
    return root


def speaker_id(rel_path):
    return Path(rel_path).name.split("_nohash_")[0]


def class_names(root):
    return sorted(d.name for d in Path(root).iterdir() if d.is_dir() and not d.name.startswith("_"))


def official_splits(root):
    """Returns {split: [(relative_path, label_idx), ...]}, classes."""
    root = Path(root)
    classes = class_names(root)
    idx = {c: i for i, c in enumerate(classes)}
    val = set(root.joinpath("validation_list.txt").read_text().split())
    test = set(root.joinpath("testing_list.txt").read_text().split())

    splits = {"train": [], "val": [], "test": []}
    for c in classes:
        for f in sorted((root / c).glob("*.wav")):
            rel = f"{c}/{f.name}"
            split = "val" if rel in val else "test" if rel in test else "train"
            splits[split].append((rel, idx[c]))
    return splits, classes


def load_pcm(path):
    # Speech Commands is 16-bit mono PCM at 16 kHz
    with wave.open(str(path), "rb") as w:
        assert w.getframerate() == SAMPLE_RATE and w.getnchannels() == 1 and w.getsampwidth() == 2, path
        return np.frombuffer(w.readframes(w.getnframes()), dtype="<i2")


def load_wav(path):
    # float in [-1, 1], like torchaudio.load
    return torch.from_numpy(load_pcm(path).astype(np.float32) / 32768.0)


def pcm_to_float(pcm):
    return pcm.float() / 32768.0


class SpeechCommandsPCM(Dataset):
    def __init__(self, root, items):
        self.root, self.items = Path(root), items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        rel, label = self.items[i]
        pcm = torch.from_numpy(load_pcm(self.root / rel)[:CLIP_SAMPLES].copy())
        return torch.nn.functional.pad(pcm, (0, CLIP_SAMPLES - len(pcm))), label


def background_noise_bank(root, part="train", holdout=0.2, hop=CLIP_SAMPLES // 2):
    # Cuts the long _background_noise_ recordings into overlapping 1 s segments: [S, 16000] int16.
    # Each recording is split in time: the first 80% feeds training augmentation ("train"), the last
    # 20% is kept for the noisy evaluation ("test"), so evaluation never replays noise seen in training.
    segs = []
    for f in sorted(Path(root, "_background_noise_").glob("*.wav")):
        pcm = torch.from_numpy(load_pcm(f).copy())
        cut = int(len(pcm) * (1 - holdout))
        pcm = pcm[:cut] if part == "train" else pcm[cut:]
        segs += [pcm[s:s + CLIP_SAMPLES] for s in range(0, len(pcm) - CLIP_SAMPLES + 1, hop)]
    return torch.stack(segs)


def build_cache(root, cache_dir, num_workers=4):
    """Stores each split as int16 waveforms [N, 16000] (lossless, ~2.7 GB for train), plus the noise bank.
    Features are computed on the GPU per batch, after waveform augmentation."""
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    splits, classes = official_splits(root)

    for split, items in splits.items():
        out = cache_dir / f"{split}.pt"
        if out.exists():
            continue
        loader = DataLoader(SpeechCommandsPCM(root, items), batch_size=1024, num_workers=num_workers)
        pcm = torch.empty(len(items), CLIP_SAMPLES, dtype=torch.int16)
        y = torch.empty(len(items), dtype=torch.long)
        n = 0
        for p, labels in loader:
            pcm[n:n + len(p)], y[n:n + len(p)] = p, labels
            n += len(p)
        torch.save({"pcm": pcm, "y": y, "files": [r for r, _ in items], "classes": classes}, out)
        print(f"{split}: {tuple(pcm.shape)} int16")

    for part, name in (("train", "noise.pt"), ("test", "noise_test.pt")):
        noise_out = cache_dir / name
        if not noise_out.exists():
            bank = background_noise_bank(root, part)
            torch.save(bank, noise_out)
            print(f"noise bank ({part}): {tuple(bank.shape)} segments")


def load_cache(cache_dir, split):
    return torch.load(Path(cache_dir) / f"{split}.pt", weights_only=False)

In [ ]:
%%writefile /kaggle/working/tat/prepare_data.py
# prepare_data.py
# Phase 0: download Speech Commands v2, check the official split, cache 1 s int16 waveforms + noise bank.
import argparse
from collections import Counter

from dataset import EXPECTED_SIZES, build_cache, download_speech_commands, official_splits, speaker_id


def check_splits(root):
    splits, classes = official_splits(root)
    assert len(classes) == 35, f"expected 35 words, found {len(classes)}: {classes}"

    speakers = {s: {speaker_id(r) for r, _ in items} for s, items in splits.items()}
    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
        shared = speakers[a] & speakers[b]
        assert not shared, f"{len(shared)} speakers in both {a} and {b}"

    for s, items in splits.items():
        per_class = Counter(label for _, label in items)
        print(f"{s:>5}: {len(items):6d} clips  {len(speakers[s]):5d} speakers  "
              f"{len(per_class)} classes  (min/max per class {min(per_class.values())}/{max(per_class.values())})")
        if len(items) != EXPECTED_SIZES[s]:
            print(f"  WARNING: expected {EXPECTED_SIZES[s]} clips for {s}")
    print("Splits are speaker-disjoint.")


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--root", default="data/speech_commands_v2")
    p.add_argument("--cache-dir", default="data/cache")
    p.add_argument("--num-workers", type=int, default=4)
    args = p.parse_args()

    download_speech_commands(args.root)
    check_splits(args.root)
    build_cache(args.root, args.cache_dir, num_workers=args.num_workers)

In [ ]:
%%writefile /kaggle/working/tat/ast_reference.py
# ast_reference.py
# Loads the authors' fine-tuned AST (MIT/ast-finetuned-speech-commands-v2, via HF transformers) into
# TinyAudioTransformer. If our modules compute the same function as AST, logits match to float precision;
# any architectural misunderstanding (patch order, token pooling, LayerNorm placement, attention layout)
# shows up as a mismatch. Used by tests/test_parity_ast.py.
import torch

from model import TATConfig, TinyAudioTransformer

HF_AST_SC = "MIT/ast-finetuned-speech-commands-v2"


def config_from_hf(hf_cfg):
    assert hf_cfg.frequency_stride == hf_cfg.time_stride, "TAT uses one stride for both axes"
    return TATConfig(
        n_mels=hf_cfg.num_mel_bins, n_frames=hf_cfg.max_length,
        patch_size=hf_cfg.patch_size, patch_stride=hf_cfg.time_stride,
        embed_dim=hf_cfg.hidden_size, depth=hf_cfg.num_hidden_layers, num_heads=hf_cfg.num_attention_heads,
        mlp_ratio=hf_cfg.intermediate_size / hf_cfg.hidden_size, num_classes=hf_cfg.num_labels,
        dropout=0.0, num_prefix_tokens=2, head_norm=True, ln_eps=hf_cfg.layer_norm_eps,
    )


# transformers renamed AST's modules over versions; rewrite older spellings to the 5.x ones.
# Order matters: attention.output.dense must be renamed before the MLP's output.dense.
_RENAMES = [
    (".encoder.layer.", ".layers."),
    (".attention.attention.query.", ".attention.q_proj."),
    (".attention.attention.key.", ".attention.k_proj."),
    (".attention.attention.value.", ".attention.v_proj."),
    (".attention.output.dense.", ".attention.o_proj."),
    (".intermediate.dense.", ".mlp.fc1."),
    (".output.dense.", ".mlp.fc2."),
]


class _TrackedDict(dict):
    # Records which tensors were read, so a mapping that silently skips weights fails loudly
    def __init__(self, *a):
        super().__init__(*a)
        self.used = set()

    def __getitem__(self, k):
        self.used.add(k)
        return super().__getitem__(k)


def _to_current_names(sd):
    out = _TrackedDict()
    for k, v in sd.items():
        for old, new in _RENAMES:
            k = k.replace(old, new)
        out[k] = v
    return out


def map_hf_state_dict(sd, depth):
    sd = _to_current_names(sd)
    e = "audio_spectrogram_transformer.embeddings."
    out = {
        "prefix_tokens": torch.cat([sd[e + "cls_token"], sd[e + "distillation_token"]], dim=1),
        "pos_embed": sd[e + "position_embeddings"],
        "patch_embed.proj.weight": sd[e + "patch_embeddings.projection.weight"],
        "patch_embed.proj.bias": sd[e + "patch_embeddings.projection.bias"],
        "norm.weight": sd["audio_spectrogram_transformer.layernorm.weight"],
        "norm.bias": sd["audio_spectrogram_transformer.layernorm.bias"],
        "head_norm.weight": sd["classifier.layernorm.weight"],
        "head_norm.bias": sd["classifier.layernorm.bias"],
        "head.weight": sd["classifier.dense.weight"],
        "head.bias": sd["classifier.dense.bias"],
    }
    for i in range(depth):
        h, o = f"audio_spectrogram_transformer.layers.{i}.", f"blocks.{i}."
        for p in ("weight", "bias"):
            qkv = [sd[h + f"attention.{n}_proj.{p}"] for n in "qkv"]
            out[o + f"attn.in_proj_{p}"] = torch.cat(qkv, dim=0)
            out[o + f"attn.out_proj.{p}"] = sd[h + f"attention.o_proj.{p}"]
            out[o + f"norm1.{p}"] = sd[h + f"layernorm_before.{p}"]
            out[o + f"norm2.{p}"] = sd[h + f"layernorm_after.{p}"]
            out[o + f"mlp.0.{p}"] = sd[h + f"mlp.fc1.{p}"]
            out[o + f"mlp.3.{p}"] = sd[h + f"mlp.fc2.{p}"]
    unused = set(sd) - sd.used
    assert not unused, f"AST tensors not mapped: {sorted(unused)[:5]}"
    return out


def load_ast_into_tat(name=HF_AST_SC):
    """Returns (our model with AST weights, the HF model, the HF feature extractor)."""
    from transformers import ASTFeatureExtractor, ASTForAudioClassification

    hf = ASTForAudioClassification.from_pretrained(name).eval()
    fe = ASTFeatureExtractor.from_pretrained(name)
    cfg = config_from_hf(hf.config)

    ours = TinyAudioTransformer(cfg).eval()
    missing, unexpected = ours.load_state_dict(map_hf_state_dict(hf.state_dict(), cfg.depth), strict=False)
    assert not unexpected, unexpected
    assert set(missing) == {"norm_in.mean", "norm_in.std"}, missing
    ours.norm_in.set_stats(fe.mean, fe.std)
    return ours, hf, fe

In [ ]:
%%writefile /kaggle/working/tat/augment.py
# augment.py
# Training-time augmentations, batched on GPU.
# Waveform stage (shift, mixup, background noise) runs before fbank, because mixing sounds is additive
# in the waveform, not in log-mel. SpecAugment runs after fbank, on the spectrogram it was designed for.
import torch


def _per_clip(v, x):
    # [B] → broadcastable against x ([B, N] or [B, T, F])
    return v.view(-1, *([1] * (x.dim() - 1)))


def time_shift(wav, max_shift):
    # wav: [B, N]. Each clip moves by its own integer in [-max_shift, max_shift]; vacated samples are zeros.
    # Teaches the model that *when* the word starts within the 1 s window doesn't matter.
    B, N = wav.shape
    shift = torch.randint(-max_shift, max_shift + 1, (B, 1), device=wav.device)
    src = torch.arange(N, device=wav.device).unsqueeze(0) - shift          # out[t] = wav[t - shift]
    valid = (src >= 0) & (src < N)
    return torch.where(valid, wav.gather(1, src.clamp(0, N - 1)), torch.zeros((), device=wav.device))


def mixup(x, targets, alpha, p):
    # A fraction p of clips is blended with a random partner: x = λ·x1 + (1-λ)·x2, and so are their labels.
    # λ ~ Beta(alpha, alpha); AST's speech-commands recipe uses p=0.6 and Beta(10, 10) (λ near 0.5).
    B = x.shape[0]
    a = torch.tensor(float(alpha), device=x.device)
    lam = torch.distributions.Beta(a, a).sample((B,))
    lam = torch.where(torch.rand(B, device=x.device) < p, lam, torch.ones_like(lam))
    perm = torch.randperm(B, device=x.device)
    x = _per_clip(lam, x) * x + _per_clip(1 - lam, x) * x[perm]
    targets = lam[:, None] * targets + (1 - lam[:, None]) * targets[perm]
    return x, targets


def add_background_noise(wav, noise_bank, snr_db=(0.0, 20.0), p=0.8):
    # Adds a random 1 s segment of real background noise (_background_noise_) at a random SNR.
    # SNR (dB) = 10·log10(P_signal / P_noise): 20 dB is faint noise, 0 dB is noise as loud as the speech.
    B = wav.shape[0]
    seg = noise_bank[torch.randint(len(noise_bank), (B,), device=wav.device)]
    snr = torch.empty(B, 1, device=wav.device).uniform_(*snr_db)
    apply = (torch.rand(B, 1, device=wav.device) < p).to(wav.dtype)
    return wav + apply * (mix_at_snr(wav, seg, snr) - wav)


def mix_at_snr(wav, noise, snr_db):
    # wav, noise: [B, N]; snr_db: [B, 1] or a number. Scales each noise row so that
    # 10·log10(P_wav / P_scaled_noise) = snr_db exactly, then adds it.
    p_sig = wav.pow(2).mean(1, keepdim=True).clamp_min(1e-10)
    p_noise = noise.pow(2).mean(1, keepdim=True).clamp_min(1e-10)
    return wav + torch.sqrt(p_sig / (p_noise * 10 ** (torch.as_tensor(snr_db, device=wav.device) / 10))) * noise


def spec_augment(x, freqm, timem, value, valid_frames=None):
    # x: [B, T, F]. Blanks one band of mel bins and one band of frames per clip, widths uniform in
    # [0, freqm] / [0, timem] (same sampling as torchaudio's FrequencyMasking/TimeMasking, used by AST).
    # Forces the model to recognise a word even when part of its spectrum or duration is missing.
    # value = the train mean, so masked cells become exactly 0 after the model's normalisation.
    # valid_frames: time masks stay inside the real audio (98 of 128 frames for 1 s); AST also masks the
    # zero padding, which wastes ~1/4 of its time masks.
    B, T, F = x.shape
    for width_max, length, dim in ((freqm, F, 2), (timem, valid_frames or T, 1)):
        if width_max <= 0:
            continue
        w = torch.randint(0, width_max + 1, (B, 1), device=x.device)
        start = (torch.rand(B, 1, device=x.device) * (length - w + 1)).long()
        pos = torch.arange(x.shape[dim], device=x.device).unsqueeze(0)
        band = (pos >= start) & (pos < start + w)                         # [B, T or F]
        x = x.masked_fill(band[:, None, :] if dim == 2 else band[:, :, None], value)
    return x


def smooth_labels(targets, eps):
    # Label smoothing: the true class gets 1-eps, the rest share eps. Discourages over-confident logits.
    return targets * (1 - eps) + eps / targets.shape[1]


def soft_cross_entropy(logits, targets):
    # Cross-entropy against a probability vector (needed once labels are mixed or smoothed)
    return -(targets * torch.log_softmax(logits, dim=1)).sum(1).mean()

In [ ]:
%%writefile /kaggle/working/tat/metrics.py
# metrics.py
import math

import torch


def accuracy(logits, y):
    return (logits.argmax(1) == y).float().mean().item() * 100


def expected_calibration_error(logits, y, n_bins=15):
    # Bin predictions by confidence; ECE = Σ (bin share) · |accuracy in bin − mean confidence in bin|.
    # 0 means "when the model says 90%, it is right 90% of the time".
    probs = logits.float().softmax(1)
    conf, pred = probs.max(1)
    correct = (pred == y).float()
    edges = torch.linspace(0, 1, n_bins + 1, device=logits.device)
    ece = torch.zeros((), device=logits.device)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.float().mean() * (correct[m].mean() - conf[m].mean()).abs()
    return ece.item() * 100


def confusion_matrix(preds, y, n):
    cm = torch.zeros(n, n, dtype=torch.long)
    cm.index_put_((y.cpu(), preds.cpu()), torch.ones_like(y.cpu()), accumulate=True)
    return cm


def mcnemar_p(correct_a, correct_b):
    # Exact two-sided McNemar test on paired predictions (same test clips): only disagreements count
    n10 = int((correct_a & ~correct_b).sum())
    n01 = int((~correct_a & correct_b).sum())
    n, k = n10 + n01, min(n10, n01)
    if n == 0:
        return 1.0
    return min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n)

In [ ]:
%%writefile /kaggle/working/tat/train.py
# train.py
# Trains on cached 1 s waveforms. Per batch, on GPU:
#   pcm → [time shift → mixup → background noise] → kaldi fbank → [SpecAugment] → model
# Augmentations are switched on by name with --aug, so each one can be measured on its own.
# Picks the checkpoint on val, then evaluates on test exactly once.
#   python train.py --model tat --aug none                 # Phase-1-style baseline
#   python train.py --model tat --aug all                  # every augmentation
#   python train.py --model tat --overfit-batch 64         # sanity check: must reach 100% on one batch
import argparse
import json
import math
import random
import sys
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

import augment
from baselines import CNNBaseline, CNNConfig
from dataset import load_cache, pcm_to_float
from features import fbank, num_frames
from metrics import accuracy, confusion_matrix, expected_calibration_error
from model import TATConfig, TinyAudioTransformer

AUGS = ("shift", "mixup", "noise", "specaug", "ls")
N_MELS, N_FRAMES = 128, 128


def parse_args(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--model", choices=["tat", "cnn"], default="tat")
    p.add_argument("--cache-dir", default="data/cache")
    p.add_argument("--out-dir", default="runs/tat")
    p.add_argument("--epochs", type=int, default=30)
    p.add_argument("--batch-size", type=int, default=256)
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--weight-decay", type=float, default=0.05)
    p.add_argument("--warmup-epochs", type=float, default=2)
    p.add_argument("--grad-clip", type=float, default=1.0)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--no-amp", action="store_true")
    p.add_argument("--limit", type=int, default=0, help="use only N clips per split (smoke tests)")
    p.add_argument("--overfit-batch", type=int, default=0, help="train on one batch of N clips until 100%%")
    # Augmentation: comma-separated subset of AUGS, or "none" / "all"
    p.add_argument("--aug", default="none")
    p.add_argument("--shift-ms", type=float, default=100)
    p.add_argument("--mixup-prob", type=float, default=0.6)
    p.add_argument("--mixup-alpha", type=float, default=10.0)
    p.add_argument("--noise-prob", type=float, default=0.8)
    p.add_argument("--snr-min", type=float, default=0.0)
    p.add_argument("--snr-max", type=float, default=20.0)
    p.add_argument("--freqm", type=int, default=24)
    p.add_argument("--timem", type=int, default=24)
    p.add_argument("--label-smoothing", type=float, default=0.1)
    # TinyAudioTransformer size
    p.add_argument("--embed-dim", type=int, default=128)
    p.add_argument("--depth", type=int, default=4)
    p.add_argument("--heads", type=int, default=4)
    p.add_argument("--mlp-ratio", type=float, default=4.0)
    p.add_argument("--patch-stride", type=int, default=10)
    p.add_argument("--dropout", type=float, default=0.1)
    args = p.parse_args(argv)

    args.aug = set(AUGS) if args.aug == "all" else set() if args.aug == "none" else set(args.aug.split(","))
    unknown = args.aug - set(AUGS)
    if unknown:
        p.error(f"unknown --aug {sorted(unknown)}; choose from {AUGS}")
    return args


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def build_model(args, num_classes):
    if args.model == "cnn":
        return CNNBaseline(CNNConfig(num_classes=num_classes))
    return TinyAudioTransformer(TATConfig(
        embed_dim=args.embed_dim, depth=args.depth, num_heads=args.heads, mlp_ratio=args.mlp_ratio,
        patch_stride=args.patch_stride, dropout=args.dropout, num_classes=num_classes))


def load_split(cache_dir, split, device, limit=0):
    d = load_cache(cache_dir, split)
    if limit:
        idx = torch.randperm(len(d["y"]), generator=torch.Generator().manual_seed(0))[:limit]
        d["pcm"], d["y"] = d["pcm"][idx], d["y"][idx]
        d["files"] = [d["files"][i] for i in idx.tolist()]
    # Whole split fits on a T4 as int16 (train ≈ 2.7 GB), so batching is just indexing
    d["pcm"], d["y"] = d["pcm"].to(device), d["y"].to(device)
    return d


def features(pcm):
    # fbank stays in fp32 (outside autocast): power spectra span many orders of magnitude
    return fbank(pcm_to_float(pcm), N_MELS, N_FRAMES)


@torch.no_grad()
def all_features(pcm, batch_size=1024):
    return torch.cat([features(pcm[i:i + batch_size]).half() for i in range(0, len(pcm), batch_size)])


@torch.no_grad()
def feature_stats(pcm, batch_size=1024):
    # Mean/std of clean train features, padding included (the model sees padding too); streamed to save memory
    s = s2 = n = 0.0
    for i in range(0, len(pcm), batch_size):
        x = features(pcm[i:i + batch_size]).double()
        s, s2, n = s + x.sum().item(), s2 + x.pow(2).sum().item(), n + x.numel()
    mean = s / n
    return mean, (s2 / n - mean ** 2) ** 0.5


@torch.no_grad()
def predict(model, x, batch_size, amp):
    model.eval()
    out = []
    for i in range(0, len(x), batch_size):
        with torch.autocast(x.device.type, dtype=torch.float16, enabled=amp):
            out.append(model(x[i:i + batch_size].float()).float())
    return torch.cat(out)


def augmented_batch(pcm, y, args, noise_bank, mean, num_classes):
    wav = pcm_to_float(pcm)
    targets = nn.functional.one_hot(y, num_classes).float()
    if "shift" in args.aug:
        wav = augment.time_shift(wav, int(args.shift_ms * 16))
    if "mixup" in args.aug:
        wav, targets = augment.mixup(wav, targets, args.mixup_alpha, args.mixup_prob)
    if "noise" in args.aug:
        wav = augment.add_background_noise(wav, noise_bank, (args.snr_min, args.snr_max), args.noise_prob)
    x = fbank(wav, N_MELS, N_FRAMES)
    if "specaug" in args.aug:
        x = augment.spec_augment(x, args.freqm, args.timem, value=mean, valid_frames=num_frames(pcm.shape[1]))
    if "ls" in args.aug:
        targets = augment.smooth_labels(targets, args.label_smoothing)
    return x, targets


def overfit_one_batch(model, train, args, device, amp, max_steps=1000):
    g = torch.Generator().manual_seed(args.seed)
    idx = torch.randperm(len(train["y"]), generator=g)[:args.overfit_batch].to(device)
    x, y = features(train["pcm"][idx]), train["y"][idx]
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=0.0)
    for step in range(1, max_steps + 1):
        model.train()
        loss = nn.functional.cross_entropy(model(x), y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        if step % 10 == 0:
            acc = accuracy(predict(model, x, len(x), amp), y)
            print(f"step {step:4d}  loss {loss.item():.4f}  acc {acc:.1f}%", flush=True)
            if acc == 100.0:
                print(f"PASS: memorised {len(y)} clips in {step} steps")
                return True
    print(f"FAIL: did not reach 100% on {len(y)} clips in {max_steps} steps")
    return False


def main(argv=None):
    args = parse_args(argv)
    seed_everything(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = device.type == "cuda" and not args.no_amp
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    train = load_split(args.cache_dir, "train", device, args.limit)
    classes = train["classes"]
    model = build_model(args, len(classes)).to(device)
    mean, std = feature_stats(train["pcm"])
    model.norm_in.set_stats(mean, std)  # clean train-split stats only
    n_params = sum(p.numel() for p in model.parameters())
    print(f"{args.model}: {n_params / 1e6:.3f}M params, {len(train['y'])} train clips, device {device}, "
          f"aug {sorted(args.aug) or 'none'}, feature mean {mean:.3f} std {std:.3f}")

    if args.overfit_batch:
        sys.exit(0 if overfit_one_batch(model, train, args, device, amp) else 1)

    noise_bank = pcm_to_float(torch.load(Path(args.cache_dir) / "noise.pt").to(device)) if "noise" in args.aug else None
    val = load_split(args.cache_dir, "val", device, args.limit)
    val_x = all_features(val["pcm"])  # clean eval features, computed once

    # No weight decay on biases, norms, tokens and position embeddings
    decay = [p for n, p in model.named_parameters() if p.ndim >= 2 and "pos_embed" not in n and "prefix" not in n]
    no_decay = [p for n, p in model.named_parameters() if not (p.ndim >= 2 and "pos_embed" not in n and "prefix" not in n)]
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": args.weight_decay},
                             {"params": no_decay, "weight_decay": 0.0}], lr=args.lr)

    steps_per_epoch = math.ceil(len(train["y"]) / args.batch_size)
    total, warmup = args.epochs * steps_per_epoch, int(args.warmup_epochs * steps_per_epoch)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / max(1, warmup) if s < warmup
                                              else 0.5 * (1 + math.cos(math.pi * (s - warmup) / max(1, total - warmup))))
    scaler = torch.amp.GradScaler(device.type, enabled=amp)
    g = torch.Generator(device=device).manual_seed(args.seed)

    history, best_val, best_epoch = [], -1.0, -1
    t0 = time.time()
    for epoch in range(1, args.epochs + 1):
        model.train()
        perm = torch.randperm(len(train["y"]), device=device, generator=g)
        loss_sum, correct, seen = 0.0, 0, 0
        for i in range(0, len(perm), args.batch_size):
            idx = perm[i:i + args.batch_size]
            y = train["y"][idx]
            x, targets = augmented_batch(train["pcm"][idx], y, args, noise_bank, mean, len(classes))
            with torch.autocast(device.type, dtype=torch.float16, enabled=amp):
                logits = model(x)
            loss = augment.soft_cross_entropy(logits.float(), targets)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
            scaler.step(opt)
            scaler.update()
            sched.step()

            loss_sum += loss.item() * len(y)
            correct += (logits.argmax(1) == y).sum().item()  # vs the dominant (unmixed) label
            seen += len(y)

        val_logits = predict(model, val_x, 1024, amp)
        val_acc = accuracy(val_logits, val["y"])
        val_nll = nn.functional.cross_entropy(val_logits, val["y"]).item()
        history.append({"epoch": epoch, "train_loss": loss_sum / seen, "train_acc": 100 * correct / seen,
                        "val_acc": val_acc, "val_nll": val_nll, "lr": sched.get_last_lr()[0],
                        "time_s": time.time() - t0})
        print(f"epoch {epoch:3d}  loss {loss_sum / seen:.4f}  train {100 * correct / seen:5.2f}%  "
              f"val {val_acc:5.2f}%  val_nll {val_nll:.3f}  lr {sched.get_last_lr()[0]:.2e}  {time.time() - t0:.0f}s",
              flush=True)
        if val_acc > best_val:
            best_val, best_epoch = val_acc, epoch
            torch.save({"model": args.model, "config": model.config_dict(), "state_dict": model.state_dict(),
                        "classes": classes, "epoch": epoch, "val_acc": val_acc}, out_dir / "best.pt")

    # Test once, with the val-selected checkpoint
    del train
    model.load_state_dict(torch.load(out_dir / "best.pt", map_location=device)["state_dict"])
    val_logits = predict(model, val_x, 1024, amp)
    test = load_split(args.cache_dir, "test", device, args.limit)
    logits = predict(model, all_features(test["pcm"]), 1024, amp)
    preds, y = logits.argmax(1).cpu(), test["y"].cpu()
    cm = confusion_matrix(preds, y, len(classes))
    per_class = (cm.diag().float() / cm.sum(1).clamp(min=1) * 100).tolist()

    results = {
        "model": args.model, "config": model.config_dict(), "params": n_params, "seed": args.seed,
        "aug": sorted(args.aug), "best_epoch": best_epoch, "best_val_acc": best_val,
        "val_ece": expected_calibration_error(val_logits, val["y"]),
        "test_acc": accuracy(logits, test["y"]), "test_ece": expected_calibration_error(logits, test["y"]),
        "test_nll": nn.functional.cross_entropy(logits, test["y"]).item(),
        "per_class_test_acc": dict(zip(classes, per_class)), "confusion_matrix": cm.tolist(),
        "train_time_s": time.time() - t0, "args": {**vars(args), "aug": sorted(args.aug)},
    }
    (out_dir / "results.json").write_text(json.dumps(results, indent=2))
    (out_dir / "history.json").write_text(json.dumps(history, indent=2))
    torch.save({"logits": logits.cpu(), "y": y, "files": test["files"], "classes": classes}, out_dir / "test_logits.pt")
    print(f"\nbest val {best_val:.2f}% (epoch {best_epoch})  →  TEST {results['test_acc']:.2f}%  "
          f"ECE {results['test_ece']:.2f}%")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/tat/sweep.py
# sweep.py
# Runs a named set of experiments × 3 seeds, scheduled over all GPUs.
#   phase2:  each augmentation alone vs. none vs. all
#   phase2b: combine the two winners (shift + SpecAugment), then add the others back one at a time
# Finished runs (results.json present) are skipped, so re-running resumes an interrupted sweep.
#   python sweep.py --sweep phase2b --cache-dir /tmp/sc_cache --out-root runs_phase2b
import argparse
import itertools
import os
import subprocess
import sys
import time
from pathlib import Path

import torch

EXPERIMENTS = {
    "tat_none":    "--model tat --aug none",
    "tat_shift":   "--model tat --aug shift",
    "tat_noise":   "--model tat --aug noise",
    "tat_mixup":   "--model tat --aug mixup",
    "tat_specaug": "--model tat --aug specaug",
    "tat_ls":      "--model tat --aug ls",
    "tat_all":     "--model tat --aug all",
    "cnn_none":    "--model cnn --aug none",
    "cnn_all":     "--model cnn --aug all",
    "tat_shift_specaug":       "--model tat --aug shift,specaug",
    "tat_shift_specaug_mixup": "--model tat --aug shift,specaug,mixup",
    "tat_shift_specaug_ls":    "--model tat --aug shift,specaug,ls",
    "tat_shift_specaug_noise": "--model tat --aug shift,specaug,noise",
}

SWEEPS = {
    "phase2": ["tat_none", "tat_shift", "tat_noise", "tat_mixup", "tat_specaug", "tat_ls", "tat_all",
               "cnn_none", "cnn_all"],
    # none and noise are re-run in the same session so 2b is self-contained (and as a reproducibility check)
    "phase2b": ["tat_none", "tat_noise", "tat_shift_specaug", "tat_shift_specaug_mixup",
                "tat_shift_specaug_ls", "tat_shift_specaug_noise"],
}


def last_line(log):
    try:
        lines = log.read_text().strip().splitlines()
        return lines[-1] if lines else "starting"
    except FileNotFoundError:
        return "starting"


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--cache-dir", default="data/cache")
    p.add_argument("--sweep", choices=SWEEPS, default="phase2b")
    p.add_argument("--out-root", default="runs_phase2b")
    p.add_argument("--seeds", type=int, nargs="+", default=[0, 1, 2])
    p.add_argument("--only", nargs="+", default=None, help="subset of experiment names")
    p.add_argument("--per-gpu", type=int, default=2, help="concurrent runs per GPU (tiny models leave a T4 idle)")
    p.add_argument("--report-every", type=int, default=120, help="seconds between progress prints")
    p.add_argument("extra", nargs=argparse.REMAINDER, help="passed to train.py after --, e.g. -- --epochs 1")
    args = p.parse_args()
    extra = args.extra[1:] if args.extra[:1] == ["--"] else args.extra

    names = args.only or SWEEPS[args.sweep]
    queue = []
    for name, seed in itertools.product(names, args.seeds):
        out = Path(args.out_root) / f"{name}_s{seed}"
        if (out / "results.json").exists():
            print(f"skip {out.name} (done)")
            continue
        queue.append((out, EXPERIMENTS[name].split() + ["--seed", str(seed)]))

    n_gpu = torch.cuda.device_count()
    slots = [str(g) for g in range(n_gpu) for _ in range(args.per_gpu)] if n_gpu else [None]
    running, failed = {}, []
    print(f"{len(queue)} runs, {len(slots)} slots ({n_gpu} GPUs × {args.per_gpu})", flush=True)

    t0, last_report = time.time(), 0.0
    while queue or running:
        for slot in slots:
            if slot not in running and queue:
                out, cmd = queue.pop(0)
                out.mkdir(parents=True, exist_ok=True)
                env = {**os.environ, **({"CUDA_VISIBLE_DEVICES": slot} if slot is not None else {})}
                full = [sys.executable, "train.py", "--cache-dir", args.cache_dir, "--out-dir", str(out), *cmd, *extra]
                running[slot] = (out, subprocess.Popen(full, stdout=open(out / "train.log", "w"),
                                                       stderr=subprocess.STDOUT, env=env))
                print(f"[{(time.time() - t0) / 60:5.1f} min] start {out.name} on GPU {slot}", flush=True)
        for slot, (out, proc) in list(running.items()):
            if proc.poll() is not None:
                del running[slot]
                status = "done" if proc.returncode == 0 else f"FAILED (exit {proc.returncode})"
                if proc.returncode != 0:
                    failed.append(out.name)
                print(f"[{(time.time() - t0) / 60:5.1f} min] {status} {out.name}: {last_line(out / 'train.log')}",
                      flush=True)
        if time.time() - last_report > args.report_every and running:
            last_report = time.time()
            for out, _ in running.values():
                print(f"    {out.name}: {last_line(out / 'train.log')}", flush=True)
        time.sleep(5)

    print(f"\nsweep finished in {(time.time() - t0) / 60:.1f} min; failed: {failed or 'none'}")
    sys.exit(1 if failed else 0)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/tat/analyze.py
# analyze.py
# Summarises a sweep: mean ± std over seeds, and each config's change vs. its "none" baseline,
# paired by seed (seed s of a config is compared with seed s of the baseline).
#   python analyze.py runs_phase2            → prints a table and writes runs_phase2/summary.md
import json
import re
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch

from metrics import mcnemar_p


def load_runs(root):
    runs = defaultdict(dict)
    for f in sorted(Path(root).glob("*/results.json")):
        m = re.fullmatch(r"(.+)_s(\d+)", f.parent.name)
        if m:
            runs[m.group(1)][int(m.group(2))] = json.loads(f.read_text())
    return runs


def test_logits(root, name, seed):
    return torch.load(Path(root) / f"{name}_s{seed}" / "test_logits.pt", weights_only=False)


def correct_vector(root, name, seed):
    d = test_logits(root, name, seed)
    return (d["logits"].argmax(1) == d["y"]).numpy()


def overconfidence(root, name, seed):
    # mean confidence − accuracy, in points: > 0 overconfident, < 0 underconfident (ECE alone has no sign)
    d = test_logits(root, name, seed)
    conf, pred = d["logits"].float().softmax(1).max(1)
    return (conf.mean() - (pred == d["y"]).float().mean()).item() * 100


def fmt(xs):
    return f"{np.mean(xs):.2f} ± {np.std(xs, ddof=1) if len(xs) > 1 else 0:.2f}"


def parent_of(name, names):
    # The longest other config whose name this one extends: tat_shift_specaug_mixup → tat_shift_specaug.
    # Falls back to <model>_none.
    prefixes = [n for n in names if name.startswith(n + "_")]
    return max(prefixes, key=len) if prefixes else name.split("_")[0] + "_none"


def compare(root, runs, name, ref):
    # Paired by seed: Δ test acc (mean, min, max), seeds better, per-seed McNemar p
    common = [s for s in sorted(runs[name]) if s in runs.get(ref, {})]
    if name == ref or not common:
        return ["—", "—", "—"]
    deltas = [runs[name][s]["test_acc"] - runs[ref][s]["test_acc"] for s in common]
    ps = [mcnemar_p(correct_vector(root, name, s), correct_vector(root, ref, s)) for s in common]
    return [f"{np.mean(deltas):+.2f} (min {min(deltas):+.2f}, max {max(deltas):+.2f})",
            f"{sum(d > 0 for d in deltas)}/{len(deltas)}", ", ".join(f"{p:.2g}" for p in ps)]


def summarise(root):
    runs = load_runs(root)
    header = ("| config | seeds | params | val acc | test acc | test ECE | conf − acc | Δ test vs none (paired) "
              "| seeds better | McNemar p (per seed) | parent | Δ vs parent | seeds better | McNemar p |")
    lines = [header, "|" + "---|" * 14]
    for name, by_seed in runs.items():
        seeds = sorted(by_seed)
        r0 = by_seed[seeds[0]]
        row = [name, str(len(seeds)), f"{r0['params'] / 1e6:.2f}M",
               fmt([by_seed[s]["best_val_acc"] for s in seeds]),
               fmt([by_seed[s]["test_acc"] for s in seeds]),
               fmt([by_seed[s]["test_ece"] for s in seeds]),
               f"{np.mean([overconfidence(root, name, s) for s in seeds]):+.1f}"]
        base, parent = name.split("_")[0] + "_none", parent_of(name, runs)
        row += compare(root, runs, name, base)
        row += [parent, *compare(root, runs, name, parent)] if parent != base else ["—", "—", "—", "—"]
        lines.append("| " + " | ".join(row) + " |")

    table = "\n".join(lines)
    note = ("\nΔ is paired by seed. A real effect should be positive for every seed and clearly larger than "
            "the seed-to-seed std of the baseline. McNemar p < 0.05 means that seed's two models disagree "
            "in a way unlikely to be chance. conf − acc > 0 is overconfident, < 0 underconfident. "
            "\"parent\" is the config this one extends (e.g. shift_specaug for shift_specaug_mixup).\n")
    Path(root, "summary.md").write_text(table + "\n" + note)
    return table + "\n" + note


if __name__ == "__main__":
    print(summarise(sys.argv[1] if len(sys.argv) > 1 else "runs_phase2"))

In [ ]:
%%writefile /kaggle/working/tat/eval_noisy.py
# eval_noisy.py
# Robustness check: re-scores every trained checkpoint on val and test with *held-out* background noise
# (noise_test.pt, never used in training) mixed in at fixed SNRs. Each clip gets one fixed noise segment,
# the same for every SNR and every model, so comparisons between models are paired like the clean ones.
# Decisions (e.g. which noise setting to keep) should be made on val; test is for reporting.
#   python eval_noisy.py runs_phase2b --cache-dir /tmp/sc_cache   → noisy.json per run + noisy_summary.md
import argparse
import json
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch

from augment import mix_at_snr
from baselines import CNNBaseline, CNNConfig
from dataset import load_cache, pcm_to_float
from features import fbank
from metrics import accuracy
from model import TATConfig, TinyAudioTransformer
from train import N_FRAMES, N_MELS, all_features, predict

NOISE_SEED = {"val": 1, "test": 2}


def load_model(path, device):
    ck = torch.load(path, map_location=device, weights_only=False)
    if ck["model"] == "cnn":
        model = CNNBaseline(CNNConfig(**{**ck["config"], "channels": tuple(ck["config"]["channels"])}))
    else:
        model = TinyAudioTransformer(TATConfig(**ck["config"]))
    model.load_state_dict(ck["state_dict"])
    return model.to(device).eval()


@torch.no_grad()
def noisy_features(pcm, noise_bank, snr_db, seed, batch_size=1024):
    # The generator depends only on the split, so clip i always gets the same segment; only the SNR varies
    idx = torch.randint(len(noise_bank), (len(pcm),), generator=torch.Generator().manual_seed(seed))
    out = []
    for i in range(0, len(pcm), batch_size):
        wav = pcm_to_float(pcm[i:i + batch_size])
        wav = mix_at_snr(wav, noise_bank[idx[i:i + batch_size]].to(wav.device), float(snr_db))
        out.append(fbank(wav, N_MELS, N_FRAMES).half())  # no clipping: we want the exact SNR
    return torch.cat(out)


def fmt(xs):
    return f"{np.mean(xs):.2f} ± {np.std(xs, ddof=1) if len(xs) > 1 else 0:.2f}"


def summarise(root, levels):
    runs = defaultdict(dict)
    for f in sorted(Path(root).glob("*/noisy.json")):
        m = re.fullmatch(r"(.+)_s(\d+)", f.parent.name)
        if m:
            runs[m.group(1)][int(m.group(2))] = json.loads(f.read_text())
    out = []
    for split in ("val", "test"):
        out += [f"\n### {split}: accuracy with held-out noise (mean ± std over seeds; Δ vs none, paired by seed)\n",
                "| config | " + " | ".join(f"{l} dB" if l != "clean" else "clean" for l in levels) + " |",
                "|---|" + "---|" * len(levels)]
        for name, by_seed in runs.items():
            base = runs.get(name.split("_")[0] + "_none", {})
            cells = []
            for l in levels:
                accs = [by_seed[s][split][l] for s in sorted(by_seed)]
                cell = fmt(accs)
                common = [s for s in by_seed if s in base]
                if base and not name.endswith("_none") and common:
                    cell += f" ({np.mean([by_seed[s][split][l] - base[s][split][l] for s in common]):+.2f})"
                cells.append(cell)
            out.append(f"| {name} | " + " | ".join(cells) + " |")
    text = "\n".join(out) + "\n"
    Path(root, "noisy_summary.md").write_text(text)
    return text


def main():
    p = argparse.ArgumentParser()
    p.add_argument("runs_root")
    p.add_argument("--cache-dir", default="data/cache")
    p.add_argument("--snrs", type=float, nargs="+", default=[20, 10, 5, 0])
    args = p.parse_args()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    amp = device == "cuda"

    run_dirs = sorted(d.parent for d in Path(args.runs_root).glob("*/best.pt"))
    models = {d: load_model(d / "best.pt", device) for d in run_dirs}
    noise_bank = pcm_to_float(torch.load(Path(args.cache_dir) / "noise_test.pt"))
    levels = ["clean"] + [f"{s:g}" for s in args.snrs]
    print(f"{len(models)} checkpoints, {len(noise_bank)} held-out noise segments, levels {levels}")

    scores = {d: {} for d in run_dirs}
    for split in ("val", "test"):
        data = load_cache(args.cache_dir, split)
        pcm, y = data["pcm"].to(device), data["y"].to(device)
        for level in levels:
            x = all_features(pcm) if level == "clean" else noisy_features(pcm, noise_bank, float(level), NOISE_SEED[split])
            for d, model in models.items():
                scores[d].setdefault(split, {})[level] = accuracy(predict(model, x, 1024, amp), y)
            print(f"{split} {level:>5}: done", flush=True)
        del pcm, x

    for d in run_dirs:
        (d / "noisy.json").write_text(json.dumps(scores[d], indent=2))
        # Sanity check: the clean test score must reproduce what train.py reported for this checkpoint
        reported = json.loads((d / "results.json").read_text())["test_acc"]
        if abs(scores[d]["test"]["clean"] - reported) > 0.05:
            print(f"WARNING {d.name}: clean test {scores[d]['test']['clean']:.2f} ≠ reported {reported:.2f}")
    print(summarise(args.runs_root, levels))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/tat/tests/conftest.py
import os
import sys
from pathlib import Path

import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))


@pytest.fixture(scope="session")
def sc_root():
    # Set SC_ROOT to the extracted Speech Commands v2 folder to run data tests
    root = os.environ.get("SC_ROOT")
    if not root or not Path(root, "testing_list.txt").exists():
        pytest.skip("SC_ROOT not set to an extracted Speech Commands v2 folder")
    return Path(root)


def hf_or_skip():
    # Parity tests need transformers + network access to the Hugging Face Hub (or a warm cache)
    pytest.importorskip("transformers")
    try:
        from ast_reference import load_ast_into_tat
        return load_ast_into_tat()
    except OSError as e:
        pytest.skip(f"cannot load AST from the Hub: {e}")

In [ ]:
%%writefile /kaggle/working/tat/tests/test_features.py
import math

import pytest
import torch

from features import SAMPLE_RATE, fbank


def _test_signal(seconds=1.0, seed=0):
    # Chirp + noise: exercises the whole spectrum, unlike pure noise
    g = torch.Generator().manual_seed(seed)
    t = torch.arange(int(seconds * SAMPLE_RATE)) / SAMPLE_RATE
    chirp = 0.3 * torch.sin(2 * math.pi * (100 + 3000 * t) * t)
    return chirp + 0.05 * torch.randn(len(t), generator=g)


@pytest.mark.parametrize("seconds,real_frames", [(0.3, 28), (1.0, 98), (1.5, 148)])
def test_fixed_output_shape(seconds, real_frames):
    out = fbank(_test_signal(seconds), n_mels=128, n_frames=128)
    assert out.shape == (128, 128)
    if real_frames < 128:
        assert torch.all(out[real_frames:] == 0), "padding must be zeros, as in AST"
        assert torch.all(out[:real_frames] != 0)


def test_batch_matches_single():
    wavs = torch.stack([_test_signal(seed=s) for s in range(3)])
    batched = fbank(wavs, n_frames=128)
    for i in range(3):
        torch.testing.assert_close(batched[i], fbank(wavs[i], n_frames=128))


def test_matches_torchaudio_kaldi():
    kaldi = pytest.importorskip("torchaudio.compliance.kaldi")
    wav = _test_signal()
    ref = kaldi.fbank(wav.unsqueeze(0), sample_frequency=SAMPLE_RATE, window_type="hanning", num_mel_bins=128)
    torch.testing.assert_close(fbank(wav), ref, atol=1e-3, rtol=1e-4)


def test_matches_hf_ast_feature_extractor():
    transformers = pytest.importorskip("transformers")
    try:
        fe = transformers.ASTFeatureExtractor.from_pretrained("MIT/ast-finetuned-speech-commands-v2")
    except OSError as e:
        pytest.skip(f"cannot reach the Hub: {e}")
    wav = _test_signal()
    hf = fe(wav.numpy(), sampling_rate=SAMPLE_RATE, return_tensors="pt").input_values[0]
    hf_raw = hf * (2 * fe.std) + fe.mean  # undo AST normalisation; ours lives inside the model
    torch.testing.assert_close(fbank(wav, 128, fe.max_length), hf_raw, atol=1e-3, rtol=1e-4)

In [ ]:
%%writefile /kaggle/working/tat/tests/test_model.py
import pytest
import torch
import torch.nn as nn

from baselines import CNNBaseline, CNNConfig
from model import TATConfig, TinyAudioTransformer


@pytest.mark.parametrize("stride,grid", [(10, (12, 12)), (16, (8, 8))])
def test_patch_count_matches_conv_output(stride, grid):
    m = TinyAudioTransformer(TATConfig(patch_stride=stride))
    assert m.patch_embed.grid == grid
    with torch.no_grad():
        conv_out = m.patch_embed.proj(torch.zeros(1, 1, 128, 128))
    assert tuple(conv_out.shape[2:]) == grid
    assert m.pos_embed.shape[1] == grid[0] * grid[1] + m.cfg.num_prefix_tokens


@pytest.mark.parametrize("prefix,head_norm", [(1, False), (2, True)])
@pytest.mark.parametrize("batch", [1, 3])
def test_forward_shape(prefix, head_norm, batch):
    m = TinyAudioTransformer(TATConfig(num_prefix_tokens=prefix, head_norm=head_norm)).eval()
    assert m(torch.randn(batch, 128, 128)).shape == (batch, 35)


@pytest.mark.parametrize("shape", [(2, 63, 128), (2, 1, 128, 64), (2, 128, 64)])
def test_rejects_wrong_input_shape(shape):
    # The old pipeline produced 63 frames for a 1 s clip and crashed inside the pos_embed add
    with pytest.raises(ValueError, match="expected input"):
        TinyAudioTransformer()(torch.randn(*shape))


def test_eval_is_deterministic():
    m = TinyAudioTransformer().eval()
    x = torch.randn(2, 128, 128)
    torch.testing.assert_close(m(x), m(x))


def test_init_scale():
    m = TinyAudioTransformer()
    assert m.pos_embed.std().item() == pytest.approx(0.02, rel=0.2)
    assert m.prefix_tokens.abs().max().item() < 0.1  # trunc_normal_ bounds are absolute (±2), so this is ~4σ


@pytest.mark.parametrize("model", ["tat", "cnn"])
def test_overfits_one_batch(model):
    # A model that cannot memorise 32 examples has a bug (wiring, masking, gradient flow)
    torch.manual_seed(0)
    if model == "tat":
        m = TinyAudioTransformer(TATConfig(embed_dim=64, depth=2, num_heads=2, patch_stride=16, dropout=0.0))
    else:
        m = CNNBaseline(CNNConfig(dropout=0.0))  # 2-layer version is too small to memorise pure noise through avg-pool
    m.norm_in.set_stats(0.0, 1.0)
    x, y = torch.randn(32, 128, 128), torch.randint(0, 35, (32,))
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    for _ in range(300):
        m.train()
        loss = nn.functional.cross_entropy(m(x), y)
        opt.zero_grad()
        loss.backward()
        opt.step()
    m.eval()
    with torch.no_grad():
        assert (m(x).argmax(1) == y).all()

In [ ]:
%%writefile /kaggle/working/tat/tests/test_splits.py
from dataset import EXPECTED_SIZES, official_splits, speaker_id


def test_official_split_sizes(sc_root):
    splits, classes = official_splits(sc_root)
    assert len(classes) == 35
    assert {s: len(v) for s, v in splits.items()} == EXPECTED_SIZES


def test_speaker_disjoint(sc_root):
    splits, _ = official_splits(sc_root)
    speakers = {s: {speaker_id(r) for r, _ in v} for s, v in splits.items()}
    assert not speakers["train"] & speakers["val"]
    assert not speakers["train"] & speakers["test"]
    assert not speakers["val"] & speakers["test"]


def test_every_split_has_every_class(sc_root):
    # The old 80/10/10 slice of a label-sorted list put only the last label(s) in test
    splits, classes = official_splits(sc_root)
    for items in splits.values():
        assert {label for _, label in items} == set(range(len(classes)))


def test_no_background_noise_clips(sc_root):
    splits, _ = official_splits(sc_root)
    assert not any(r.startswith("_") for items in splits.values() for r, _ in items)

In [ ]:
%%writefile /kaggle/working/tat/tests/test_parity_ast.py
# Our architecture loaded with the authors' AST weights must compute the same function as their model.
import os
from pathlib import Path

import pytest
import torch

from conftest import hf_or_skip
from features import SAMPLE_RATE, fbank


@pytest.fixture(scope="module")
def models():
    return hf_or_skip()


def test_logits_match_on_random_input(models):
    ours, hf, fe = models
    g = torch.Generator().manual_seed(0)
    hf_in = torch.randn(2, fe.max_length, 128, generator=g)          # normalised, as HF expects
    raw = hf_in * (2 * fe.std) + fe.mean                             # what our model expects
    with torch.no_grad():
        torch.testing.assert_close(ours(raw), hf(hf_in).logits, atol=1e-3, rtol=1e-3)


def test_end_to_end_on_audio(models):
    # Waveform → our fbank → our model  vs  waveform → HF extractor → HF model
    ours, hf, fe = models
    root = os.environ.get("SC_ROOT")
    if root and Path(root, "testing_list.txt").exists():
        from dataset import load_wav
        rels = Path(root, "testing_list.txt").read_text().split()[::1000][:8]
        wavs = [load_wav(Path(root, r)) for r in rels]
        labels = [r.split("/")[0] for r in rels]
    else:
        g = torch.Generator().manual_seed(0)
        wavs, labels = [0.1 * torch.randn(SAMPLE_RATE, generator=g) for _ in range(2)], None

    with torch.no_grad():
        ours_logits = torch.stack([ours(fbank(w, 128, fe.max_length)[None])[0] for w in wavs])
        hf_logits = hf(**fe([w.numpy() for w in wavs], sampling_rate=SAMPLE_RATE, return_tensors="pt")).logits
    torch.testing.assert_close(ours_logits, hf_logits, atol=1e-3, rtol=1e-3)

    if labels:  # the reference itself should be right on real test clips, or we are comparing to garbage
        id2label = hf.config.id2label
        preds = [id2label[i] for i in ours_logits.argmax(1).tolist()]
        assert sum(p == l for p, l in zip(preds, labels)) >= len(labels) - 1, list(zip(preds, labels))

In [ ]:
%%writefile /kaggle/working/tat/tests/test_augment.py
import math

import pytest
import torch

import augment
from metrics import expected_calibration_error, mcnemar_p


@pytest.fixture(autouse=True)
def seed():
    torch.manual_seed(0)


def test_time_shift_moves_content_and_zero_fills():
    wav = torch.randn(16, 200)
    out = augment.time_shift(wav, max_shift=20)
    for w, o in zip(wav, out):
        matches = [s for s in range(-20, 21)
                   if torch.equal(o, torch.roll(w, s) * ((torch.arange(200) - s >= 0) & (torch.arange(200) - s < 200)))]
        assert matches, "each output row must be its input shifted by some s in [-20, 20], zero-filled"


def test_time_shift_zero_is_identity():
    wav = torch.randn(4, 100)
    torch.testing.assert_close(augment.time_shift(wav, 0), wav)


@pytest.mark.parametrize("snr", [0.0, 10.0, 20.0])
def test_noise_hits_requested_snr(snr):
    wav, bank = torch.randn(8, 16000) * 0.1, torch.randn(5, 16000)
    out = augment.add_background_noise(wav, bank, (snr, snr), p=1.0)
    noise = out - wav
    measured = 10 * torch.log10(wav.pow(2).mean(1) / noise.pow(2).mean(1))
    torch.testing.assert_close(measured, torch.full((8,), snr), atol=1e-3, rtol=0)


def test_noise_p0_is_identity():
    wav = torch.randn(8, 16000)
    torch.testing.assert_close(augment.add_background_noise(wav, torch.randn(3, 16000), p=0.0), wav)


def test_mixup_is_convex_and_labels_sum_to_one():
    x = torch.rand(32, 100)
    t = torch.nn.functional.one_hot(torch.randint(0, 35, (32,)), 35).float()
    xm, tm = augment.mixup(x, t, alpha=10.0, p=1.0)
    torch.testing.assert_close(tm.sum(1), torch.ones(32))
    assert (xm >= x.min() - 1e-6).all() and (xm <= x.max() + 1e-6).all()
    assert not torch.equal(xm, x)


def test_mixup_p0_is_identity():
    x, t = torch.rand(8, 50), torch.eye(8)
    xm, tm = augment.mixup(x, t, 10.0, p=0.0)
    torch.testing.assert_close(xm, x)
    torch.testing.assert_close(tm, t)


def test_spec_augment_masks_one_contiguous_band_per_axis():
    x = torch.randn(64, 128, 128)
    out = augment.spec_augment(x, freqm=24, timem=24, value=99.0)
    masked = out == 99.0
    for m in masked:
        freq_cols = m.all(0).nonzero().flatten()   # mel bins masked across all frames
        time_rows = m.all(1).nonzero().flatten()   # frames masked across all bins
        for band in (freq_cols, time_rows):
            assert len(band) <= 24
            if len(band):
                assert band.max() - band.min() + 1 == len(band), "band must be contiguous"
        # nothing outside those two bands is touched
        expected = torch.zeros_like(m)
        expected[:, freq_cols] = True
        expected[time_rows, :] = True
        assert torch.equal(m, expected)


def test_spec_augment_time_mask_stays_in_real_frames():
    x = torch.randn(256, 128, 128)
    masked_rows = (augment.spec_augment(x, 0, 24, value=99.0, valid_frames=98) == 99.0).all(2)
    assert masked_rows.any(), "some time masks expected"
    assert not masked_rows[:, 98:].any(), "padding frames must never be masked"


def test_soft_ce_equals_ce_on_one_hot():
    logits, y = torch.randn(10, 35), torch.randint(0, 35, (10,))
    soft = augment.soft_cross_entropy(logits, torch.nn.functional.one_hot(y, 35).float())
    torch.testing.assert_close(soft, torch.nn.functional.cross_entropy(logits, y))


def test_label_smoothing_matches_pytorch():
    logits, y = torch.randn(10, 35), torch.randint(0, 35, (10,))
    t = augment.smooth_labels(torch.nn.functional.one_hot(y, 35).float(), 0.1)
    torch.testing.assert_close(augment.soft_cross_entropy(logits, t),
                               torch.nn.functional.cross_entropy(logits, y, label_smoothing=0.1))


def test_ece_zero_when_calibrated_and_large_when_overconfident():
    y = torch.zeros(1000, dtype=torch.long)
    # Model says class 0 with 80% confidence and is right 80% of the time → ECE ≈ 0
    logit0 = math.log(0.8 / 0.2)
    logits = torch.tensor([[logit0, 0.0]]).repeat(1000, 1)
    y[800:] = 1
    assert expected_calibration_error(logits, y) < 0.5
    # Same 80% accuracy but 99.9% confident → ECE ≈ 19.9 points
    logits = torch.tensor([[math.log(999), 0.0]]).repeat(1000, 1)
    assert expected_calibration_error(logits, y) == pytest.approx(19.9, abs=0.2)


def test_mcnemar():
    a = torch.ones(100, dtype=torch.bool)
    assert mcnemar_p(a.numpy(), a.numpy()) == 1.0
    b = a.clone()
    b[:20] = False                       # 20 disagreements, all favouring a
    assert mcnemar_p(a.numpy(), b.numpy()) < 1e-4

In [ ]:
%%writefile /kaggle/working/tat/tests/test_noisy.py
import wave

import numpy as np
import torch

import augment
from analyze import parent_of
from baselines import CNNBaseline
from dataset import background_noise_bank
from eval_noisy import load_model, noisy_features
from model import TATConfig, TinyAudioTransformer


def test_noise_holdout_never_overlaps_training_noise(tmp_path):
    # First 80% of the recording is +1000, last 20% is −1000: each bank must see only its own part
    (tmp_path / "_background_noise_").mkdir()
    pcm = np.full(160000, 1000, dtype=np.int16)
    pcm[128000:] = -1000
    with wave.open(str(tmp_path / "_background_noise_" / "hum.wav"), "wb") as w:
        w.setnchannels(1), w.setsampwidth(2), w.setframerate(16000)
        w.writeframes(pcm.tobytes())
    train, test = background_noise_bank(tmp_path, "train"), background_noise_bank(tmp_path, "test")
    assert len(train) and len(test)
    assert (train == 1000).all() and (test == -1000).all()


def test_mix_at_snr_is_exact():
    wav, noise = torch.randn(4, 16000), torch.randn(4, 16000) * 3
    out = augment.mix_at_snr(wav, noise, 5.0)
    measured = 10 * torch.log10(wav.pow(2).mean(1) / (out - wav).pow(2).mean(1))
    torch.testing.assert_close(measured, torch.full((4,), 5.0), atol=1e-3, rtol=0)


def test_noisy_features_are_repeatable_and_get_worse_with_lower_snr():
    pcm = (torch.randn(6, 16000) * 3000).to(torch.int16)
    bank = torch.randn(10, 16000)
    a, b = noisy_features(pcm, bank, 10, seed=1), noisy_features(pcm, bank, 10, seed=1)
    assert torch.equal(a, b), "every model must be scored on identical noisy clips"
    clean = noisy_features(pcm, bank, 60, seed=1).float()
    err10 = (a.float() - clean).abs().mean()
    err0 = (noisy_features(pcm, bank, 0, seed=1).float() - clean).abs().mean()
    assert err0 > err10


def test_load_model_round_trip(tmp_path):
    x = torch.randn(2, 128, 128)
    for m in (TinyAudioTransformer(TATConfig(depth=1)), CNNBaseline()):
        m.eval()
        kind = "cnn" if isinstance(m, CNNBaseline) else "tat"
        torch.save({"model": kind, "config": m.config_dict(), "state_dict": m.state_dict()}, tmp_path / "best.pt")
        torch.testing.assert_close(load_model(tmp_path / "best.pt", "cpu")(x), m(x))


def test_parent_is_longest_extended_config():
    names = ["tat_none", "tat_shift", "tat_shift_specaug", "tat_shift_specaug_mixup"]
    assert parent_of("tat_shift_specaug_mixup", names) == "tat_shift_specaug"
    assert parent_of("tat_shift_specaug", names) == "tat_shift"
    assert parent_of("tat_noise", names) == "tat_none"

## Data: official split, int16 waveform cache, and two noise banks
`noise.pt` (first 80% of each background recording) is for training; `noise_test.pt` (last 20%) is only for the noisy evaluation.

In [ ]:
DATA, CACHE = "/tmp/sc_v2", "/tmp/sc_cache"
!python prepare_data.py --root {DATA} --cache-dir {CACHE} --num-workers 4

## Tests (all should pass, none skipped)

In [ ]:
!SC_ROOT={DATA} python -m pytest tests -q -rs -p no:cacheprovider

## Learnability on the new waveform pipeline (both must print PASS)

In [ ]:
!python train.py --model tat --cache-dir {CACHE} --overfit-batch 128 | tail -1
!python train.py --model cnn --cache-dir {CACHE} --overfit-batch 128 | tail -1

## Listen and look: what 20 / 10 / 5 / 0 dB of held-out noise sounds like
0 dB means the noise has as much power as the speech. Watch the word's shape fade into the background in the spectrograms.

In [ ]:
import matplotlib.pyplot as plt
import torch
from IPython.display import Audio, display

from augment import mix_at_snr
from dataset import load_cache, pcm_to_float
from features import fbank

WORD = "yes"
test = load_cache(CACHE, "test")
held_out = pcm_to_float(torch.load(f"{CACHE}/noise_test.pt"))   # never used in training
clip = pcm_to_float(test["pcm"][[int((test["y"] == test["classes"].index(WORD)).nonzero()[0])]])
seg = held_out[[len(held_out) // 2]]

waves = {"clean": clip, **{f"{snr} dB": mix_at_snr(clip, seg, float(snr)) for snr in (20, 10, 5, 0)}}
fig, axes = plt.subplots(1, len(waves), figsize=(3.6 * len(waves), 3.2), sharey=True)
for ax, (title, w) in zip(axes, waves.items()):
    ax.imshow(fbank(w, 128, 128)[0].T, origin="lower", aspect="auto", cmap="magma")
    ax.set_title(f"'{WORD}' {title}", fontsize=9)
    ax.set_xlabel("frame (10 ms)")
axes[0].set_ylabel("mel bin")
plt.tight_layout()
plt.show()
for title, w in waves.items():
    print(title)
    display(Audio(w[0].numpy(), rate=16000))

## The sweep
Progress prints every 2 minutes. If the session dies, re-run this cell: finished runs are skipped.

In [ ]:
!python sweep.py --sweep phase2b --cache-dir {CACHE} --out-root /kaggle/working/runs_phase2b --per-gpu 2

## Results on the clean test (Δ vs none, and vs the config each one extends)

In [ ]:
!python analyze.py /kaggle/working/runs_phase2b

## Robustness: every checkpoint on held-out noise (val for decisions, test for reporting)

In [ ]:
!python eval_noisy.py /kaggle/working/runs_phase2b --cache-dir {CACHE}

In [ ]:
!cd /kaggle/working && zip -qr phase2b_results.zip runs_phase2b && ls -lh phase2b_results.zip

## Send back
Download `phase2b_results.zip` from the Output tab, put it in the repo root, and tell me. I'll analyse it and add the findings to `docs/LEARNING_LOG.md`.